In [ ]:
# @title 0) 저장소 클론·pip 설치 (로컬에서는 생략 가능)
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK_REL = Path("src") / "mindscopex_analysis" / "__init__.py"


def find_repo_root(start: Path | None = None) -> Path | None:
    candidate = (start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / MARK_REL).is_file():
            return path
    return None


root = find_repo_root()
if root is None:
    workdir = Path(os.environ.get("COLAB_REPO_DIR", "/content/mindscopex_analysis"))
    if (workdir / MARK_REL).is_file():
        print(f"already cloned -> git pull: {workdir}")
        subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=False)
        root = workdir
    else:
        workdir.parent.mkdir(parents=True, exist_ok=True)
        if workdir.exists():
            shutil.rmtree(workdir)
        print(f"cloning {REPO_URL} -> {workdir}")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(workdir)])
        root = workdir
else:
    print(f"local repo: {root}")

os.environ["MINDSCOPEX_ROOT"] = str(root.resolve())
os.chdir(root)
print("cwd =", os.getcwd())
print("MINDSCOPEX_ROOT =", os.environ["MINDSCOPEX_ROOT"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("pip install -e . done")


# Bat and Ball Lure Feature Ablation

목표: bat-and-ball 문제에서 함정 답 `10 cents` 쪽 logprob을 밀어주는 Qwen-Scope feature 후보를 찾고, 해당 feature의 decoder direction을 제거했을 때 `logprob(10 cents) - logprob(5 cents)` margin이 어떻게 바뀌는지 확인합니다.

## 실험 방식

1. 마지막 prompt token residual을 Qwen-Scope SAE로 encode해서 active feature 후보를 뽑습니다.
2. 각 후보 feature마다 `feature_value * W_dec[:, feature_id]`를 같은 위치의 residual stream에서 뺍니다.
3. teacher forcing으로 `10 cents`와 `5 cents`의 answer logprob을 각각 계산합니다.
4. `baseline_margin - ablated_margin`이 큰 feature를 함정 답에 기여한 후보로 봅니다.

In [ ]:
from pathlib import Path
import os
import sys

root = Path(os.environ.get("MINDSCOPEX_ROOT", Path.cwd())).resolve()
if not (root / "src" / "mindscopex_analysis" / "__init__.py").is_file():
    for candidate in [root, *root.parents]:
        if (candidate / "src" / "mindscopex_analysis" / "__init__.py").is_file():
            root = candidate
            break
    else:
        raise RuntimeError("Could not find repository root. Run the clone cell first.")

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(root)


In [ ]:
import torch
from IPython.display import display

from mindscopex_analysis import (
    BAT_BALL_CASE,
    DEFAULT_MODEL_ID,
    DEFAULT_QWEN_SCOPE_REPO_ID,
    active_prompt_features,
    answer_logprob_margin,
    capture_layer_residuals,
    default_sae_device,
    dtype_from_name,
    feature_handle_from_result,
    load_qwen_language_model,
    load_qwen_scope_sae,
    rank_lure_feature_effects,
    recommended_dtype_name,
    save_feature_handle,
)


In [ ]:
MODEL_ID = DEFAULT_MODEL_ID
SAE_REPO_ID = DEFAULT_QWEN_SCOPE_REPO_ID
DTYPE = recommended_dtype_name()
SAE_DEVICE = default_sae_device()
SAE_DTYPE = DTYPE

CASE = BAT_BALL_CASE
PROMPT = CASE.prompt
CORRECT_ANSWER = CASE.correct_answer
LURE_ANSWER = CASE.lure_answer
FEATURE_CACHE = root / "outputs" / "candidates" / "bat_ball_top_feature.json"

# 넓게 보려면 [6, 14, 21, 27]로 늘리세요.
# 후보 수를 늘리면 forward pass가 2 * 후보 수 * layer 수만큼 늘어납니다.
CANDIDATE_LAYERS = [14]
TOP_N_CANDIDATES = 12
ABLATION_COEFFICIENT = 1.0

print({
    "model": MODEL_ID,
    "sae_repo": SAE_REPO_ID,
    "dtype": DTYPE,
    "sae_device": SAE_DEVICE,
    "layers": CANDIDATE_LAYERS,
    "top_n_candidates": TOP_N_CANDIDATES,
    "feature_cache": str(FEATURE_CACHE),
})


In [ ]:
lm = load_qwen_language_model(
    MODEL_ID,
    device_map="auto",
    dtype=DTYPE,
    dispatch=True,
)

print("loaded", MODEL_ID)


In [ ]:
baseline = answer_logprob_margin(
    lm,
    PROMPT,
    correct_answer=CORRECT_ANSWER,
    lure_answer=LURE_ANSWER,
)

display(baseline.as_row())
print("correct tokens:", list(zip(baseline.correct.tokens, baseline.correct.token_logprobs)))
print("lure tokens:", list(zip(baseline.lure.tokens, baseline.lure.token_logprobs)))


In [ ]:
all_results = []
candidate_rows = []

for layer in CANDIDATE_LAYERS:
    residual = capture_layer_residuals(
        lm,
        [PROMPT],
        layer,
        token_position="last",
    )
    sae = load_qwen_scope_sae(
        SAE_REPO_ID,
        layer,
        device=SAE_DEVICE,
        dtype=dtype_from_name(SAE_DTYPE),
    )
    candidates = active_prompt_features(
        residual,
        sae,
        top_n=TOP_N_CANDIDATES,
    )
    candidate_rows.extend(
        {
            "layer": layer,
            "rank": rank,
            "feature_id": feature_id,
            "feature_value": feature_value,
        }
        for rank, (feature_id, feature_value) in enumerate(candidates, start=1)
    )

    _, layer_results = rank_lure_feature_effects(
        lm,
        PROMPT,
        correct_answer=CORRECT_ANSWER,
        lure_answer=LURE_ANSWER,
        layer=layer,
        sae=sae,
        residual=residual,
        candidate_features=candidates,
        top_n_candidates=TOP_N_CANDIDATES,
        coefficient=ABLATION_COEFFICIENT,
    )
    all_results.extend(layer_results)

    del sae
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

display(candidate_rows)


In [ ]:
all_results = sorted(all_results, key=lambda item: item.margin_delta, reverse=True)
result_rows = [item.as_row() for item in all_results]
display(result_rows)

best = all_results[0]
handle = feature_handle_from_result(CASE, best)
save_feature_handle(handle, FEATURE_CACHE)

print("best layer:", best.layer)
print("best feature:", best.feature_id)
print("feature value:", best.feature_value)
print("baseline margin lure-correct:", best.baseline_margin)
print("ablated margin lure-correct:", best.ablated_margin)
print("margin delta:", best.margin_delta)
print("saved feature handle:", FEATURE_CACHE)


## 읽는 법

- `baseline_margin`: 원래 `logprob(10 cents) - logprob(5 cents)`입니다.
- `ablated_margin`: 해당 feature를 제거한 뒤의 같은 margin입니다.
- `margin_delta`: `baseline_margin - ablated_margin`입니다. 클수록 feature 제거가 함정 답의 상대 우위를 많이 낮춘 것입니다.
- `lure_logprob_delta`가 음수이면 제거 뒤 `10 cents` 자체의 logprob이 내려갔다는 뜻입니다.
- `correct_logprob_delta`가 양수이면 제거 뒤 `5 cents` logprob이 올라갔다는 뜻입니다.